In [1]:
import xarray as xr
import numpy as np
import time
from xclim.indices import standardized_precipitation_index
from google.oauth2 import service_account
from dask.distributed import Client
import coiled
from dask.diagnostics import ProgressBar

# Start timing
start_time = time.time()

# Path to service account credentials
credentials_path = 'coiled-data-e4drr.json'

# Define the required scopes for read/write access
scopes = ["https://www.googleapis.com/auth/devstorage.read_write"]

# Create credentials object with the required scope
credentials = service_account.Credentials.from_service_account_file(
    credentials_path, scopes=scopes
)

# Input CHIRPS Zarr path in GCS
zarr_path = "gs://ittseas51/ea_chirps_v20_monthly_20250415.zarr"

# Output path for results in GCS
output_gcs_path = "gs://ittseas51/ea_spi3_chirps_20250417.zarr"

# Storage options for xarray to use with GCS
storage_options = {'token': credentials}

# Create a Coiled cluster with proper scaling
print("Setting up Coiled cluster...")
cluster = coiled.Cluster(
    name="spi-calculation-dask-workers-6",  
    software="itt-jupyter-env-v20250318",
    n_workers=6,               # Increased workers for full dataset
    scheduler_vm_types=["n2-standard-4"],
    worker_vm_types="n2-standard-4",
    region="us-east1",
    arm=False,
    compute_purchase_option="spot",
    workspace='geosfm',
    worker_options={
        "security": {
            "key_path": credentials_path  # Pass credentials to workers
        }
    }
)

client = Client(cluster)
print(f"Dask dashboard: {client.dashboard_link}")

try:
    # Open the dataset with explicit chunk sizes
    print("Opening CHIRPS dataset...")
    chunk_sizes = {
        'latitude': 50,
        'longitude': 50,
        'time': 531  # Original chunk size for time
    }
    
    ds = xr.open_dataset(
        zarr_path, 
        engine='zarr', 
        chunks=chunk_sizes,  # Use explicit chunk sizes
        consolidated=False,
        storage_options=storage_options
    )
    
    print("Dataset info:")
    print(ds)
    print(f"Dataset chunking: {ds.chunks}")
    
    # Calculate memory requirements
    nbytes = ds.nbytes
    print(f"Approximate memory size of dataset: {nbytes / 1e9:.2f} GB")
    
    # Calculate SPI-3 on the full dataset
    print("Calculating SPI-3 on full dataset...")
    
    # Extract precipitation data
    precip_data = ds.precip
    precip_data.attrs['units'] = 'mm/month'
    
    # Compute SPI-3 using xclim (Dask-aware)
    spi_3 = standardized_precipitation_index(
        precip_data,
        freq="MS",         
        window=3,         
        dist="gamma",       
        method="APP",      
        cal_start='1991-01-01',
        cal_end='2018-01-01',
        fitkwargs={"floc": 0}
    )
    
    # Create a proper dataset with coordinates
    spi_result = xr.Dataset(
        {"spi3": spi_3},
        coords=ds.coords
    )
    
    # Add metadata
    spi_result.spi3.attrs.update({
        'long_name': 'Standardized Precipitation Index (3-month)',
        'units': 'unitless',
        'description': 'SPI calculated using gamma distribution with APP method, caliberation period is 1991-2018'
    })
    
    # Print information about result
    print("SPI result info:")
    print(spi_result)
    
    # Keep the same chunk structure as input
    # Get the existing chunks from the input dataset
    output_chunks = {
        'time': ds.chunks['time'][0],          # Likely 531
        'latitude': ds.chunks['latitude'][0],  # Likely 50
        'longitude': ds.chunks['longitude'][0] # Likely 50
    }
    
    print(f"Using output chunks: {output_chunks}")
    
    # Define encoding for compression and chunks
    encoding = {
        'spi3': {
            'compressor': None,
            'dtype': 'float32',
            '_FillValue': -9999.0,
            'chunks': tuple(output_chunks[dim] for dim in spi_result.spi3.dims)
        }
    }
    
    # Write to GCS
    print("Writing result to GCS...")
    with ProgressBar():
        # Apply chunking before writing
        spi_result = spi_result.chunk(output_chunks)
        
        write_job = spi_result.to_zarr(
            output_gcs_path,
            mode='w',
            encoding=encoding,
            consolidated=True,
            storage_options=storage_options,
            compute=False  # Don't compute yet, get the future
        )
        
        # Execute the write task
        future = client.compute(write_job)
        
        # Wait for completion
        future.result()
    
    # Calculate simple statistics to verify data (on a sample)
    print("Calculating statistics on a sample...")
    with ProgressBar():
        # Take a small sample for quicker stats calculation
        sample = spi_result.spi3.isel(
            time=slice(0, min(12, len(spi_result.time))),
            latitude=slice(0, min(100, len(spi_result.latitude))),
            longitude=slice(0, min(100, len(spi_result.longitude)))
        )
        
        spi_min = float(sample.min().compute())
        spi_max = float(sample.max().compute())
        spi_mean = float(sample.mean().compute())
    
    print(f"SPI-3 sample stats: min={spi_min:.3f}, max={spi_max:.3f}, mean={spi_mean:.3f}")
    
    total_time = time.time() - start_time
    print(f"Processing completed successfully in {total_time:.2f} seconds.")
    
except Exception as e:
    print(f"Error during processing: {str(e)}")
    raise
    
finally:
    # Clean up
    client.close()
    cluster.close()
    print("Coiled cluster closed.")

Setting up Coiled cluster...


Output()

Dask dashboard: https://cluster-mlovq.dask.host/z1utInf3Ue6Gc7RG/status
Opening CHIRPS dataset...
Dataset info:
<xarray.Dataset> Size: 2GB
Dimensions:    (time: 531, latitude: 1000, longitude: 1000)
Coordinates:
  * longitude  (longitude) float32 4kB 10.02 10.07 10.12 ... 59.88 59.93 59.97
  * latitude   (latitude) float32 4kB -19.98 -19.93 -19.88 ... 29.88 29.92 29.97
  * time       (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2025-03-01
Data variables:
    precip     (time, latitude, longitude) float32 2GB dask.array<chunksize=(531, 50, 50), meta=np.ndarray>
Attributes: (12/15)
    Conventions:       CF-1.6
    title:             CHIRPS Version 2.0
    history:           created by Climate Hazards Group
    version:           Version 2.0
    date_created:      2025-04-14
    creator_name:      Pete Peterson
    ...                ...
    reference:         Funk, C.C., Peterson, P.J., Landsfeld, M.F., Pedreros,...
    comments:           time variable denotes the first day of t

/opt/conda/envs/itt/lib/python3.11/site-packages/xclim/indices/stats.py:972: UserWarning: The input data is chunked on time dimension and must be fully rechunked to run `fit` on groups . Beware, this operation can significantly increase the number of tasks dask has to handle.
  da, _ = preprocess_standardized_index(da, freq=freq, window=window, **indexer)


SPI result info:
<xarray.Dataset> Size: 8GB
Dimensions:       (longitude: 1000, latitude: 1000, time: 531)
Coordinates:
  * longitude     (longitude) float32 4kB 10.02 10.07 10.12 ... 59.93 59.97
  * latitude      (latitude) float32 4kB -19.98 -19.93 -19.88 ... 29.92 29.97
    prob_of_zero  (time, latitude, longitude) float64 4GB dask.array<chunksize=(531, 50, 50), meta=np.ndarray>
  * time          (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2025-03-01
Data variables:
    spi3          (time, latitude, longitude) float64 4GB dask.array<chunksize=(531, 50, 50), meta=np.ndarray>
Using output chunks: {'time': 531, 'latitude': 50, 'longitude': 50}
Writing result to GCS...


/opt/conda/envs/itt/lib/python3.11/site-packages/zarr/core/group.py:2469: UserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(
/opt/conda/envs/itt/lib/python3.11/site-packages/zarr/api/asynchronous.py:203: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/opt/conda/envs/itt/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 376.75 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Calculating statistics on a sample...
SPI-3 sample stats: min=-8.210, max=8.210, mean=-0.823
Processing completed successfully in 761.37 seconds.
Coiled cluster closed.
